[![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-orange?logo=googlecolab)](https://colab.research.google.com/github/YOUR_USERNAME/satellite-change-detection/blob/main/notebooks/03_bce_dice_upgrade.ipynb)

# 03 BCE-Dice Upgrade

Complete the validation threshold sweep and train the first segmentation-aligned model.

**Prerequisites:** Google Colab GPU runtime, LEVIR-CD dataset access, repo files available.

**Expected runtime:** 45-75 minutes on a T4 GPU


## Step 1: Threshold Sweep

This cell completes the missing threshold sweep from the original notebook. It evaluates 50 thresholds on the entire validation split, keeps the heavy tensor math on GPU, saves the requested plots, prints a summary table, and stores `BEST_THRESHOLD`.

In [ ]:
from pathlib import Path
import torch
from src.factory import load_config, build_model
from src.dataset import create_dataloaders
from src.evaluate import threshold_sweep
from src.utils import select_device

config = load_config(Path('configs/baseline.yaml'))
config['data']['root'] = '/content/levir_data'
device = select_device('cuda')
_, _, _, _, val_loader, _ = create_dataloaders(config)
model = build_model(config).to(device)
state = torch.load('/content/siamese_baseline_best.pth', map_location=device)
model.load_state_dict(state)
thresholds = torch.linspace(0.1, config['loss']['margin'] * 1.5, 50)
sweep = threshold_sweep(model, val_loader, device, thresholds, Path('/content'), margin=float(config['loss']['margin']))
BEST_THRESHOLD = sweep['best_threshold']
print('BEST_THRESHOLD =', BEST_THRESHOLD)
# Expected output: a best threshold likely near 0.8-1.4, plus saved threshold_sweep.png and pr_curve.png in /content.


## Step 1: Qualitative Visualisation With Best Threshold

This cell re-runs a four-row qualitative grid using the best validation threshold instead of the original default threshold of 1.0.

In [ ]:
from src.utils import denormalize_image
import matplotlib.pyplot as plt

model.eval()
samples = []
for idx, batch in enumerate(val_loader):
    if idx == 4:
        break
    batch = {k: v.to(device) if hasattr(v, 'to') else v for k, v in batch.items()}
    emb_a, emb_b, dist = model(batch['image_a'], batch['image_b'])
    prob = 1.0 - (dist / config['loss']['margin']).clamp(0.0, 1.0)
    pred = (prob >= BEST_THRESHOLD).float()
    samples.append((batch['image_a'][0].cpu(), batch['image_b'][0].cpu(), batch['mask'][0,0].cpu(), prob[0,0].cpu(), pred[0,0].cpu()))

fig, axes = plt.subplots(4, 5, figsize=(16, 12))
for r, sample in enumerate(samples):
    a, b, m, p, y = sample
    axes[r,0].imshow(denormalize_image(a).permute(1,2,0))
    axes[r,1].imshow(denormalize_image(b).permute(1,2,0))
    axes[r,2].imshow(m, cmap='gray')
    axes[r,3].imshow(p, cmap='viridis', vmin=0, vmax=1)
    axes[r,4].imshow(y, cmap='gray')
    for c in range(5):
        axes[r,c].axis('off')
plt.tight_layout()
plt.show()
# Expected output: four validation examples thresholded with BEST_THRESHOLD instead of 1.0.


## Step 2: BCE-Dice Upgrade

This cell clones the baseline config in memory, enables the lightweight change head, switches the loss to BCE-Dice, and retrains the original custom-encoder architecture from scratch for a fair Step 2 comparison.

In [ ]:
from pathlib import Path
from copy import deepcopy
from src.factory import load_config
from src.train import train_from_config

step2_config = deepcopy(load_config(Path('configs/baseline.yaml')))
step2_config['data']['root'] = '/content/levir_data'
step2_config['paths']['results_dir'] = 'results/step2'
step2_config['paths']['best_checkpoint_name'] = 'siamese_bce_dice_best.pth'
step2_config['paths']['history_prefix'] = 'step2'
step2_config['model']['use_change_head'] = True
step2_config['loss'] = {
    'name': 'bce_dice',
    'bce_weight': 0.4,
    'dice_weight': 0.6,
    'smooth': 1e-6,
}
step2_config['evaluation']['threshold'] = 0.5
step2_config['inference']['threshold'] = 0.5
step2_summary = train_from_config(step2_config, config_name='step2_bce_dice_custom_encoder')
step2_summary['test_metrics']
# Expected output: noticeably higher F1 than the contrastive baseline, ideally above 0.45.


## Notebook Summary

This notebook finishes the missing threshold sweep, reuses the best threshold for baseline visualisation, and then retrains the original architecture with BCE-Dice plus a probability-producing change head so the loss is aligned with F1.